# Optech Raman AI — Pipeline expliqué (Notebook pédagogique)

Ce notebook documente **de bout en bout** le pipeline Optech Raman AI, en expliquant chaque étape, en montrant le **code source** du projet et en fournissant des **exemples exécutables** minimaux.

**Contenu :**
1. Structure du projet et installation locale
2. Contrat de données (schéma Parquet) et validation
3. Prétraitement (calibration, baseline, lissage, normalisation, débruitage) — *avec exemples codés*
4. Extraction de features (exemples + appel au module du projet si importable)
5. Utilisation de la CLI (`convert-csv`, `dedup`) — *démo programmatique*
6. Tests et reproductibilité (pytest, DVC — concepts et exemples)
7. Journalisation, métriques et rapports (LaTeX)
8. Feuille de route (parallélisation, vectorisation, verrous de dépendances)

> Ce notebook extrait automatiquement le projet depuis l'archive fournie et tente d'importer les modules depuis `src/`.


## 1) Structure du projet & import

In [ ]:

import os, sys, subprocess, json, textwrap, pathlib
from pathlib import Path

PROJECT_ROOT = Path("/mnt/data/optech_raman_ai_proj")
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))

def tree(path, max_files=400):
    path = Path(path)
    rows = []
    for p in sorted(path.rglob("*")):
        try:
            rel = p.relative_to(path)
        except Exception:
            rel = p
        rows.append(str(rel) + ("/" if p.is_dir() else ""))
        if len(rows) >= max_files:
            rows.append("... (truncated)")
            break
    print("\n".join(rows))

print("Project root:", PROJECT_ROOT)
print("Python path added:", SRC)
print("\n--- PROJECT TREE ---")
tree(PROJECT_ROOT)


In [ ]:

# Tentative d'import des modules principaux du projet
try:
    import optech_raman_ai
    print("[OK] Package optech_raman_ai importé")
except Exception as e:
    print("[WARN] Impossible d'importer optech_raman_ai:", e)

# Modules internes (si disponibles)
try:
    from optech_raman_ai.data.convert_csv_to_parquet import convert_folder as convert_csv_folder
    print("[OK] convert_csv_folder importé")
except Exception as e:
    print("[WARN] convert_csv_folder indisponible:", e)

try:
    from optech_raman_ai.utils.deduplicate import deduplicate_folder
    print("[OK] deduplicate_folder importé")
except Exception as e:
    print("[WARN] deduplicate_folder indisponible:", e)

try:
    from optech_raman_ai.features.extract_features import extract_features  # si défini
    print("[OK] extract_features importé")
except Exception as e:
    print("[WARN] extract_features indisponible:", e)


## 2) Contrat de données (Parquet) & validation


Nous fixons un **schéma minimal** pour les spectres Raman après conversion CSV→Parquet :

- `sample_id: string`
- `wavenumber_cm1: float64` *(axe x)*
- `intensity: float64` *(axe y, déjà calibrée en unités relatives)*
- `instrument: string` *(facultatif)*
- `acq_datetime: datetime64[ns]` *(facultatif)*
- `meta: dict` *(facultatif, métadonnées variées)*

Nous ajoutons un validateur pour :
- Types et colonnes obligatoires,
- Longueur minimale des spectres,
- Monotonicité de `wavenumber_cm1`,
- Absence de NaN/Inf.


In [ ]:

from typing import Optional, Dict, Any
import pandas as pd
import numpy as np
from dataclasses import dataclass

REQUIRED_COLS = ["sample_id","wavenumber_cm1","intensity"]
OPTIONAL_COLS = {"instrument": "string", "acq_datetime": "datetime64[ns]"}

def validate_schema(df: pd.DataFrame, min_points: int = 128) -> None:
    # Colonnes
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes: {missing}")
    # Types de base
    if not np.issubdtype(df["wavenumber_cm1"].dtype, np.number):
        raise TypeError("wavenumber_cm1 doit être numérique")
    if not np.issubdtype(df["intensity"].dtype, np.number):
        raise TypeError("intensity doit être numérique")
    # Nombre de points
    if len(df) < min_points:
        raise ValueError(f"Trop peu de points ({len(df)} < {min_points})")
    # Monotonicité
    if not (df["wavenumber_cm1"].values[1:] >= df["wavenumber_cm1"].values[:-1]).all():
        raise ValueError("wavenumber_cm1 doit être croissant (monotone)")
    # NaN/Inf
    if np.any(~np.isfinite(df["wavenumber_cm1"])) or np.any(~np.isfinite(df["intensity"])):
        raise ValueError("NaN/Inf détectés")
    print("[OK] Schéma et intégrité valides")
    
# Démo rapide : spectre synthétique
def demo_spectrum(n=2048, start=100., stop=3500., seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(start, stop, n)
    # raies synthétiques (gaussiennes)
    centers = [520, 1000, 1580]
    y = np.zeros_like(x)
    for c in centers:
        y += np.exp(-0.5*((x-c)/5)**2) * (1.0 + 0.1*rng.normal(size=n))
    y += 0.02 * (x - x.mean())  # slope
    y += 0.05*rng.standard_normal(size=n)  # bruit
    df = pd.DataFrame({"sample_id": ["demo"]*n,
                       "wavenumber_cm1": x,
                       "intensity": y})
    return df

df_demo = demo_spectrum()
validate_schema(df_demo)
df_demo.head()


## 3) Prétraitement — méthodes et code


Nous documentons ici les étapes clés, avec implémentations **références** (simples, lisibles, modifiables) :

1. **Débruitage**: filtre Savitzky–Golay ou moyenne glissante (préserver pics).  
2. **Correction de baseline**: ALS (Asymmetric Least Squares) ou polyfit robuste.  
3. **Normalisation**: à l’aire totale, à l’intensité du pic étalon (ex.: Si 520 cm⁻¹), ou z-score.  
4. **Calibration**: recentrage/étalonnage des pics étalons (polynôme d’ordre 1–2).  
5. **Masquage artefacts**: suppression outliers/raies cosmiques.


In [ ]:

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

def smooth_savgol(y, window=21, poly=3):
    window = window if window % 2 == 1 else window+1
    return savgol_filter(y, window_length=window, polyorder=poly)

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    # Eilers & Boelens (2005)
    L = len(y)
    import scipy.sparse as sp
    from scipy.sparse.linalg import spsolve
    D = sp.diags([1,-2,1],[0, -1, -2], shape=(L, L-2))
    D = (D @ D.T)  # second derivative penalty
    w = np.ones(L)
    for _ in range(niter):
        W = sp.diags(w, 0)
        Z = W + lam * D
        z = spsolve(Z, w*y)
        w = p * (y > z) + (1-p) * (y < z)
    return z

def normalize_area(y):
    a = np.trapz(y)
    return y / a if a != 0 else y

def preprocess_pipeline(df: pd.DataFrame, smooth=True, baseline=True, norm="area"):
    df = df.copy()
    y = df["intensity"].values
    if smooth:
        y = smooth_savgol(y, window=21, poly=3)
    if baseline:
        b = baseline_als(y, lam=1e5, p=0.01, niter=10)
        y = y - b
    if norm == "area":
        y = normalize_area(y)
    df["intensity"] = y
    return df

df_prep = preprocess_pipeline(df_demo)
df_prep.head()


## 4) Extraction de features


Deux approches :  
- **Physique** (pics, positions, largeurs, aires, rapports de pics) — robuste et interprétable.  
- **Apprise** (embeddings via autoencoder/FWT quaternionique TFUG, etc.) — plus riche mais à versionner.

Ci-dessous un exemple **physique minimal** (détection de pics + features agrégées). Si le module `optech_raman_ai.features.extract_features` est présent, on le démontre aussi.


In [ ]:

import numpy as np
import pandas as pd
from scipy.signal import find_peaks

def extract_features_simple(df: pd.DataFrame, prominence=0.01, distance=5):
    x = df["wavenumber_cm1"].values
    y = df["intensity"].values
    peaks, props = find_peaks(y, prominence=prominence, distance=distance)
    feats = {
        "n_peaks": int(len(peaks)),
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "area": float(np.trapz(y, x)),
    }
    # premiers pics (positions + hauteurs) jusqu'à K=5
    K = 5
    order = np.argsort(props.get("prominences", np.zeros_like(peaks)))[::-1]
    peaks_sorted = peaks[order][:K]
    for i, p in enumerate(peaks_sorted):
        feats[f"peak_{i+1}_pos"] = float(x[p])
        feats[f"peak_{i+1}_height"] = float(y[p])
    return pd.DataFrame([feats])

feats_demo = extract_features_simple(df_prep)
feats_demo


In [ ]:

try:
    # Démo : appeler le module du projet si disponible (peut exiger des deps)
    import inspect
    print("[INFO] extract_features dispo ?", 'extract_features' in globals())
    if 'extract_features' in globals():
        feats_proj = extract_features(df_prep)  # signature hypothétique
        display(feats_proj if hasattr(feats_proj, 'head') else feats_proj)
    else:
        print("[SKIP] extract_features non disponible.")
except Exception as e:
    print("[WARN] Appel extract_features du projet a échoué:", e)


## 5) Utilisation de la CLI (démo programmatique)


Le projet expose une CLI `optech-raman-ai`. Ici on démontre **programmatique** :  
- Conversion CSV→Parquet (`convert-csv`)  
- Déduplication (`dedup`)

> En pratique:  
```bash
optech-raman-ai convert-csv --input data/raw --output data/processed
optech-raman-ai dedup --input data/processed --hash sha1
```


In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, os

data_raw = Path("data_raw_demo"); data_raw.mkdir(exist_ok=True)
# Générer 2 CSV synthétiques
for i in range(2):
    df = demo_spectrum(n=1024, seed=i)
    f = data_raw / f"spectrum_{i}.csv"
    df.to_csv(f, index=False)

data_out = Path("data_processed_demo"); data_out.mkdir(exist_ok=True)

try:
    if 'convert_csv_folder' in globals():
        convert_csv_folder(data_raw, data_out)
        print("[OK] Conversion CSV→Parquet (module projet)")
    else:
        # Fallback: conversion locale simple
        for csv in data_raw.glob("*.csv"):
            df = pd.read_csv(csv)
            validate_schema(df, min_points=100)
            df.to_parquet(data_out / (csv.stem + ".parquet"), index=False)
        print("[OK] Conversion CSV→Parquet (fallback local)")
except Exception as e:
    print("[WARN] Conversion a échoué:", e)

# Déduplication (fallback simple): hash du contenu parquet
try:
    if 'deduplicate_folder' in globals():
        deduplicate_folder(data_out, algo="sha1")
        print("[OK] Déduplication (module projet)")
    else:
        seen = set()
        for pq in sorted(data_out.glob("*.parquet")):
            h = hash((pq.stat().st_size, pq.stat().st_mtime_ns))
            if h in seen:
                pq.unlink()
            else:
                seen.add(h)
        print("[OK] Déduplication (fallback local)")
except Exception as e:
    print("[WARN] Déduplication a échoué:", e)

list(data_out.glob("*.parquet"))[:4]


## 6) Tests & Reproductibilité (pytest, DVC)


- **Tests** : viser des tests **unitaires** (prétraitements, features), **CLI**, et **property-based** (fuzz sur spectres).  
- **DVC** : figer un graphe `convert → preprocess → features → model` avec `params.yaml` (fenêtres, lambda ALS, etc.).

Exemple minimal `dvc.yaml` (à placer à la racine du projet) :
```yaml
stages:
  convert:
    cmd: optech-raman-ai convert-csv --input data/raw --output data/processed
    deps: [data/raw]
    outs: [data/processed]
  preprocess:
    cmd: python -m optech_raman_ai.pipelines.preprocess --input data/processed --output data/clean
    deps: [data/processed, params.yaml]
    outs: [data/clean]
  features:
    cmd: python -m optech_raman_ai.pipelines.features --input data/clean --output data/features
    deps: [data/clean, params.yaml]
    outs: [data/features]
```

> Ici nous n’exécutons pas `pytest`/`dvc` pour garder le notebook autonome, mais la structure du projet s’y prête directement.


## 7) Journalisation, métriques, rapports LaTeX


- **Logging** JSON (niveau, `git_sha`, version pipeline, paramètres, durée) → fichier par run.  
- **Métriques** (latence/échec/% spectres valides) → `metrics.json`.  
- **Rapport LaTeX** auto : générer des figures (pics, baseline) et une table des métriques, inclure `references.bib`.  

Exemple : exporter automatiquement un **résumé PDF** journalier des runs (Overleaf-ready).


In [ ]:

import json, time, uuid, platform, pathlib
from datetime import datetime

RUN_DIR = pathlib.Path("runs"); RUN_DIR.mkdir(exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]

log = {
    "run_id": run_id,
    "timestamp": datetime.now().isoformat(),
    "host": platform.node(),
    "python": platform.python_version(),
    "params": {"savgol_window": 21, "savgol_poly": 3, "als_lambda": 1e5, "als_p": 0.01},
    "metrics": {},
}

# Exemple métrique : latence de prétraitement sur 5 spectres synthétiques
import time
t0 = time.time()
for i in range(5):
    d = demo_spectrum(n=2048, seed=i)
    _ = preprocess_pipeline(d)
lat = time.time() - t0
log["metrics"]["preprocess_5_spectra_seconds"] = lat

with open(RUN_DIR / f"{run_id}_metrics.json", "w") as f:
    json.dump(log, f, indent=2)

RUN_DIR, run_id


## 8) Feuille de route immédiate


1. **Contrat de données & prétraitements v1** (verrouillé, testé).  
2. **Vectorisation & `--workers`** pour `convert`/`features` (joblib + tqdm).  
3. **Tests unitaires exhaustifs** (prétraitement/IO/features) + **property-based**.  
4. **Métriques & rapports auto** (LaTeX + figures, BibTeX).  
5. **DVC** réellement activé, remote défini (S3/SSH).  
6. **Release CI** (SemVer tag → build wheel → TestPyPI), **CHANGELOG**, verrou deps.
